# V2 Phase 15 — Final 420-case benchmark (Colab GPU)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Official evaluation: **140 frozen FinQA test questions × 3 architectures = 420 cases**.

Uses **llama_cpp + Qwen3-8B Q4_K_M**, Phase 6 knowledge base, identical retrieval, and locked **T = 0.65**.

The Phase 14 9-case notebook is **engineering evidence only** — do not re-run it instead of this job.

Does **not** modify the frozen 140/40. Does **not** recalibrate T. Does **not** change RAG architectures or retrieval config. Does **not** modify V1.

## Setup

Push latest V2 (Phase 15 `scripts/run_full_benchmark.py`) to branch `main`, then run **this notebook on Colab GPU**.

Historical Colab clones used a previous development workspace (legacy launch configuration).

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/` and `threshold.lock.json` (clone or Drive `configs/phase13/`).

**Outputs:** `results/raw/phase15_benchmark/{run_id}/cases.jsonl`, checkpoints, logs, Drive copies under `MyDrive/MSc-RAG/`.

If Colab disconnects: run the **resume** cell. Do **not** start a new run from question 1.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError('Open this notebook on Colab GPU. Do not run the 420-case benchmark on the Mac.')

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'run_full_benchmark.py').is_file():
    raise FileNotFoundError(f'Phase 15 script missing at {V2_ROOT}. Push Phase 15 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base + lock file from Drive

Shared Phase 6 / Phase 8 index. Identical retrieval for all three architectures. Does **not** copy the Mac Chroma database.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
os.environ['V2_DRIVE_ROOT'] = str(DRIVE_ROOT)
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

lock_dst = V2 / 'results' / 'config' / 'threshold.lock.json'
lock_src = DRIVE_ROOT / 'configs' / 'phase13' / 'threshold.lock.json'
if not lock_dst.is_file() and lock_src.is_file():
    lock_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(lock_src, lock_dst)
    print('restored lock from Drive', lock_dst)
elif lock_dst.is_file():
    print('lock present in clone', lock_dst)
else:
    raise FileNotFoundError('threshold.lock.json missing. Push Phase 13 lock or copy it to Drive configs/phase13/.')

## 4. Index preflight + lock check (T=0.65, not recalibrated)

In [ ]:
import json
from pathlib import Path
from src.calibration.lock import load_official_lock
from src.run.benchmark import BENCHMARK_N_CASES, BENCHMARK_N_QUESTIONS, select_benchmark_questions

!PYTHONPATH=. python scripts/validate_kb_index.py
lock = load_official_lock()
print('locked T', lock['threshold'], 'source_split', lock['source_split'], 'used_frozen_test_140', lock['used_frozen_test_140'])
if float(lock['threshold']) != 0.65:
    raise RuntimeError('Expected locked T=0.65. Do not recalibrate.')
if lock.get('used_frozen_test_140') is True:
    raise RuntimeError('Lock must not use the frozen 140')
questions = select_benchmark_questions(n=BENCHMARK_N_QUESTIONS, allow_full=True)
print('frozen_test_n', len(questions), 'planned_cases', BENCHMARK_N_CASES)
if len(questions) != 140:
    raise RuntimeError('Phase 15 must use all 140 frozen test questions.')

## 5. Final 420-case benchmark (`llama_cpp`, T=0.65)

Incremental JSONL + Drive checkpoint after each case. If this cell is interrupted, use the **resume** cell — do not create a new run_id.

In [ ]:
import os
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
os.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'
!PYTHONPATH=. python scripts/run_full_benchmark.py --backend llama_cpp

## 5b. Resume after disconnect (only if section 5 did not finish)

Skips completed `{architecture}:{question_id}` keys. Retries failed cases. Does not restart from question 1. Does not re-run the Phase 14 9-case notebook.

In [ ]:
# Uncomment only after an interrupted 420-case run:
# import os
# os.environ['V2_REQUIRE_CUDA'] = '1'
# os.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'
# !PYTHONPATH=. python scripts/run_full_benchmark.py --backend llama_cpp --resume-latest

## 6. Completion summary — 420 cases, T=0.65, no duplicates

In [ ]:
import json
from pathlib import Path
from collections import Counter
from src.config import load_experiment_config

summary = Path('results/config/phase15_benchmark_summary.json')
smoke = Path('results/config/phase15_smoke_test.json')
fp = Path('results/config/phase15_runtime_fingerprint.json')
print('summary:', summary.is_file(), 'record:', smoke.is_file(), 'fingerprint:', fp.is_file())

data = json.loads(summary.read_text())
print('status:', data.get('status'))
print('phase:', data.get('phase'), 'mode:', data.get('mode'))
print('run_id:', data.get('run_id'))
print('backend:', data.get('backend'), 'device:', data.get('device'), 'gpu:', data.get('gpu'))
print('completed/failed/pending:', data.get('n_completed'), data.get('n_failed'), data.get('n_pending'))
print('n_questions/n_cases:', data.get('n_questions'), data.get('n_cases'))
print('threshold:', data.get('threshold'), 'locked:', data.get('threshold_locked'), data.get('threshold_note'))
print('drive_sync:', data.get('drive_sync'))

if data.get('device') == 'mps_capable_host':
    raise RuntimeError('This is a Mac result, not Colab GPU.')
if data.get('backend') != 'llama_cpp':
    raise RuntimeError(f'Expected llama_cpp, got {data.get("backend")}')
if data.get('threshold_locked') is not True or float(data.get('threshold')) != 0.65:
    raise RuntimeError('Phase 15 must use locked T=0.65')
if int(data.get('n_questions') or 0) != 140 or int(data.get('n_cases') or 0) != 420:
    raise RuntimeError('Phase 15 must be 140 questions × 3 architectures = 420 cases.')
if data.get('phase') != 15:
    raise RuntimeError('Expected phase=15 metadata (not the Phase 14 9-case job).')

raw = Path(data['raw_path'])
if not raw.is_file():
    raw = Path('/content/capstone-rag/V2') / data['raw_path']
cases = [json.loads(line) for line in raw.read_text().splitlines() if line.strip()]
last = {}
for case in cases:
    last[case.get('case_key')] = case
print('raw_lines:', len(cases), 'unique_keys:', len(last))
if len(last) != 420 and data.get('status') == 'PASS':
    raise RuntimeError(f'PASS requires 420 unique keys, got {len(last)}')

fields = load_experiment_config().section('storage').get('raw_result_fields', [])
missing_any = False
for case in last.values():
    missing = [f for f in fields if f not in case]
    if missing:
        missing_any = True
        print('MISSING', case.get('case_key'), missing)
        break
if missing_any:
    raise RuntimeError('Raw schema missing required fields.')

decisions = Counter((c.get('architecture'), c.get('decision')) for c in last.values())
print('decisions', dict(decisions))
print('progress n_completed', data.get('n_completed'), 'n_failed', data.get('n_failed'), 'n_pending', data.get('n_pending'))
print('schema OK; T=0.65 LOCKED; 420-case job')

## 7. Copy Phase 15 raw results, checkpoints, and logs to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
DRIVE = Path('/content/drive/MyDrive/MSc-RAG')

summary = json.loads((V2 / 'results' / 'config' / 'phase15_benchmark_summary.json').read_text())
run_id = summary['run_id']

raw_src = V2 / 'results' / 'raw' / 'phase15_benchmark' / run_id
raw_dest = DRIVE / 'results' / 'raw' / 'phase15_benchmark' / run_id
raw_dest.parent.mkdir(parents=True, exist_ok=True)
if raw_src.is_dir():
    shutil.copytree(raw_src, raw_dest, dirs_exist_ok=True)
    print('copied raw', raw_dest)

ckpt_src = V2 / 'results' / 'checkpoints' / 'phase15_benchmark'
ckpt_dest = DRIVE / 'checkpoints' / 'phase15_benchmark'
if ckpt_src.is_dir():
    shutil.copytree(ckpt_src, ckpt_dest, dirs_exist_ok=True)
    print('copied checkpoints', ckpt_dest)

log_src = V2 / 'results' / 'logs'
log_dest = DRIVE / 'logs' / 'phase15'
if log_src.is_dir():
    log_dest.mkdir(parents=True, exist_ok=True)
    for path in log_src.glob('*phase15*'):
        shutil.copy2(path, log_dest / path.name)
        print('copied log', path.name)

cfg_dest = DRIVE / 'configs' / 'phase15'
cfg_dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase15_runtime_fingerprint.json',
    'phase15_smoke_test.json',
    'phase15_benchmark_summary.json',
    'threshold.lock.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, cfg_dest / name)
        print('copied', name)
print('Drive dest root:', raw_dest)
print('final status', summary.get('status'), 'completed', summary.get('n_completed'), '/', summary.get('n_cases'))